In [12]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

c:\Users\tlsdn\anaconda3\envs\hf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
diagnosis = pd.read_csv("diagnosis.csv")
edstays = pd.read_csv("edstays.csv")
discharge = pd.read_csv("discharge.csv")

print("diagnosis shape:", diagnosis.shape)
print("edstays shape:", edstays.shape)
print("discharge shape:", discharge.shape)

diagnosis shape: (899050, 6)
edstays shape: (425087, 9)
discharge shape: (331793, 8)


In [14]:
diag_cols = set(diagnosis.columns)
ed_cols = set(edstays.columns)
dis_cols = set(discharge.columns)

print("diagnosis columns:", diagnosis.columns.tolist())
print("edstays columns:", edstays.columns.tolist())
print("discharge columns:", discharge.columns.tolist())

print("\n[Overlapping column names]")
print("diagnosis ∩ edstays:", diag_cols & ed_cols)
print("edstays ∩ discharge:", ed_cols & dis_cols)
print("diagnosis ∩ discharge:", diag_cols & dis_cols)

diagnosis columns: ['subject_id', 'stay_id', 'seq_num', 'icd_code', 'icd_version', 'icd_title']
edstays columns: ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'gender', 'race', 'arrival_transport', 'disposition']
discharge columns: ['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime', 'text']

[Overlapping column names]
diagnosis ∩ edstays: {'subject_id', 'stay_id'}
edstays ∩ discharge: {'subject_id', 'hadm_id'}
diagnosis ∩ discharge: {'subject_id'}


In [15]:
diag_ed = diagnosis.merge(edstays, on="stay_id", how="inner")
final_df = diag_ed.merge(discharge, on="hadm_id", how="inner")

print("diag_ed shape:", diag_ed.shape)
print("final_df shape:", final_df.shape)
print("unique hadm_id:", final_df["hadm_id"].nunique())

diag_ed shape: (899050, 14)
final_df shape: (305430, 21)
unique hadm_id: 154709


In [16]:
print(final_df[["stay_id", "hadm_id", "icd_code", "icd_title"]].head(10))
print(final_df[["hadm_id", "text"]].head(3))

    stay_id     hadm_id icd_code  \
0  32952584  29079034.0     4589   
1  32952584  29079034.0    07070   
2  32952584  29079034.0      V08   
3  33258284  22595853.0     5728   
4  33258284  22595853.0    78959   
5  33258284  22595853.0    07070   
6  33258284  22595853.0      V08   
7  35968195  25742920.0     5715   
8  35968195  25742920.0    78900   
9  35968195  25742920.0      V08   

                                           icd_title  
0                                    HYPOTENSION NOS  
1  UNSPECIFIED VIRAL HEPATITIS C WITHOUT HEPATIC ...  
2                         ASYMPTOMATIC HIV INFECTION  
3                           OTH SEQUELA, CHR LIV DIS  
4                                      OTHER ASCITES  
5  UNSPECIFIED VIRAL HEPATITIS C WITHOUT HEPATIC ...  
6                         ASYMPTOMATIC HIV INFECTION  
7                             CIRRHOSIS OF LIVER NOS  
8                         ABDOMINAL PAIN UNSPEC SITE  
9                         ASYMPTOMATIC HIV INFECTION 

In [17]:
grouped = final_df.groupby("hadm_id").agg({
    "text": "first",
    "icd_code": lambda x: list(set(x))
}).reset_index()

grouped["num_codes"] = grouped["icd_code"].apply(len)

print("grouped shape:", grouped.shape)
print(grouped.head())
print(grouped["num_codes"].describe())

grouped shape: (154709, 4)
      hadm_id                                               text  \
0  20000019.0   \nName:  ___             Unit No:   ___\n \nA...   
1  20000024.0   \nName:  ___              Unit No:   ___\n \n...   
2  20000057.0   \nName:  ___                      Unit No:   ...   
3  20000239.0   \nName:  ___             Unit No:   ___\n \nA...   
4  20000254.0   \nName:  ___                    Unit No:   __...   

                   icd_code  num_codes  
0      [25000, 59080, 4019]          3  
1                    [D649]          1  
2  [9597, 2449, 486, E8859]          4  
3                    [R079]          1  
4                    [K861]          1  
count    154709.000000
mean          1.972406
std           1.271101
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max           9.000000
Name: num_codes, dtype: float64


In [18]:
all_codes = [code for codes in grouped["icd_code"] for code in codes]
code_counts = Counter(all_codes)

top_k = 20
top_codes = [c for c, _ in code_counts.most_common(top_k)]

print("Top ICD codes:")
print(code_counts.most_common(top_k))

Top ICD codes:
[('4019', 10452), ('25000', 4996), ('R109', 4268), ('R0600', 4013), ('486', 3977), ('R079', 3749), ('5990', 3695), ('78650', 3684), ('J189', 3503), ('R4182', 3424), ('R531', 3310), ('78060', 3078), ('N390', 2986), ('R509', 2932), ('78097', 2801), ('2720', 2793), ('78909', 2772), ('E8889', 2707), ('I509', 2693), ('N179', 2545)]


In [19]:
def filter_codes(codes):
    return [c for c in codes if c in top_codes]

grouped["labels"] = grouped["icd_code"].apply(filter_codes)
grouped = grouped[grouped["labels"].map(len) > 0].reset_index(drop=True)

print("Filtered grouped shape:", grouped.shape)
print(grouped[["hadm_id", "labels"]].head())

Filtered grouped shape: (59563, 5)
      hadm_id         labels
0  20000019.0  [25000, 4019]
1  20000057.0          [486]
2  20000239.0         [R079]
3  20000347.0        [78060]
4  20000694.0        [R0600]


In [30]:
code2id = {c: i for i, c in enumerate(top_codes)}
id2code = {i: c for c, i in code2id.items()}

def encode_labels(code_list):
    vec = [0.0] * len(top_codes)
    for c in code_list:
        if c in code2id:
            vec[code2id[c]] = 1.0
    return vec

grouped["label_vector"] = grouped["labels"].apply(encode_labels)

print(grouped[["labels", "label_vector"]].head())

          labels                                       label_vector
0  [25000, 4019]  [1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
1          [486]  [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...
2         [R079]  [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...
3        [78060]  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
4        [R0600]  [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...


In [31]:
data_df = grouped[["text", "label_vector"]].copy()

train_df, test_df = train_test_split(
    data_df,
    test_size=0.2,
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42
)

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

train: 42885
val: 4765
test: 11913


In [32]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False)
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True), preserve_index=False)
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True), preserve_index=False)

dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label_vector'],
        num_rows: 42885
    })
    validation: Dataset({
        features: ['text', 'label_vector'],
        num_rows: 4765
    })
    test: Dataset({
        features: ['text', 'label_vector'],
        num_rows: 11913
    })
})


In [33]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"
#model_name = "bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 11913/11913 [04:01<00:00, 49.41 examples/s]


In [34]:
def add_labels(example):
    example["labels"] = [float(x) for x in example["label_vector"]]
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)
tokenized_dataset = tokenized_dataset.remove_columns(["text", "label_vector"])
tokenized_dataset.set_format("torch")

Map: 100%|██████████| 11913/11913 [00:01<00:00, 11196.26 examples/s]


In [35]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(top_codes),
    problem_type="multi_label_classification"
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [36]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
    }

In [37]:
training_args = TrainingArguments(
    output_dir="./clinical_icd_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none"
)

In [38]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_20644\991116914.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [39]:
trainer.train()

val_result = trainer.evaluate(tokenized_dataset["validation"])
print("Validation result:", val_result)

test_result = trainer.evaluate(tokenized_dataset["test"])
print("Test result:", test_result)

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,0.123900,0.136795,0.474867,0.445924,0.650494,0.373914
2,0.126500,0.129073,0.534248,0.515397,0.646431,0.455244


Validation result: {'eval_loss': 0.12907326221466064, 'eval_micro_f1': 0.5342479176874081, 'eval_macro_f1': 0.5153967410550314, 'eval_micro_precision': 0.6464311121650462, 'eval_micro_recall': 0.4552438209752839, 'eval_runtime': 119.3914, 'eval_samples_per_second': 39.911, 'eval_steps_per_second': 4.992, 'epoch': 2.0}
Test result: {'eval_loss': 0.13096433877944946, 'eval_micro_f1': 0.5312919264462158, 'eval_macro_f1': 0.5137770523298042, 'eval_micro_precision': 0.6443955202450464, 'eval_micro_recall': 0.45196374622356494, 'eval_runtime': 303.0487, 'eval_samples_per_second': 39.311, 'eval_steps_per_second': 4.917, 'epoch': 2.0}
